# Pilot Reproduksi: Multi-GIN+EU dan MEGA-GIN (framework MEGA-GNN)

Tahap 1 rencana tesis (opsi **2 METODE**): memastikan kedua backbone berjalan di Kaggle memakai framework resmi [MEGA-GNN-TMLR](https://github.com/hcagri/MEGA-GNN-TMLR) (Bilgi & Atasu, TMLR 2026) lewat fork [aanpujianto070990/MEGA-GNN-TMLR](https://github.com/aanpujianto070990/MEGA-GNN-TMLR), lalu mengukur waktu dan memori sebagai dasar anggaran 140 run.

Kedua backbone memakai pipeline yang sama persis (split, sampler, head, optimizer, hyperparameter); bedanya hanya dua baris config:

| Backbone | `dataset.add_ports` | `gnn.multi_edge_agg` |
|---|---|---|
| Multi-GIN (= Multi-GIN+EU: ports, reverse MP, ego ID, edge update) | True | False |
| MEGA-GIN (agregasi sisi paralel dua tahap) | False | True (`gin`) |

**Sebelum menjalankan di Kaggle:**
1. *Add Input* -> dataset `ealtman2019/ibm-transactions-for-anti-money-laundering-aml`.
2. *Settings -> Accelerator* -> **GPU T4** (hindari P100: arsitekturnya tidak lagi didukung build PyTorch terbaru).
3. *Settings -> Internet* -> **On** (untuk `git clone` dan `pip install`).
4. Hasil di `/kaggle/working/results_pilot` hanya tersimpan permanen kalau notebook di-*Save Version* (atau diunduh sebelum sesi berakhir). Data olahan sengaja ditaruh di `/tmp` agar tidak ikut tersimpan (beberapa GB); dibangun ulang otomatis tiap sesi (beberapa menit).

**Alur:**
- **Tahap A - smoke test**: instalasi, build `ports_cpp`, praproses HI-Small, cek statistik dataset, 1 epoch pendek kedua model.
- **Tahap B - benchmark waktu**: 4 epoch per kombinasi model x dataset; ukur waktu latih/evaluasi dan memori GPU, lalu proyeksikan biaya protokol paper vs protokol proposal (140 run).
- **Tahap C - reproduksi penuh**: satu kombinasi per sesi dengan config asli paper (400 epoch x 256 iterasi, evaluasi tiap 4 epoch), lalu bandingkan F1 dengan paper.

Angka acuan (F1 kelas ilegal, %, Tabel 16 Bilgi & Atasu): Multi-GIN 62,66 +- 1,73 (HI) / 32,21 +- 0,99 (LI); MEGA-GIN 70,83 +- 2,19 (HI) / 43,67 +- 0,55 (LI).

**Perubahan di fork dibanding upstream** (hanya kompatibilitas, tidak mengubah model/protokol):
- `format_dataset` (praproses CSV mentah) ditulis ulang vektorisasi dengan pandas: tanpa dependensi `datatable`, hasil id/urutan sama dengan loop asli.
- `torch.load(..., weights_only=False)` untuk data olahan dan checkpoint (default `torch>=2.6` menolak objek `HeteroData`).
- Import `datatable` di `jodie_dataset.py` dibuat opsional; import `scipy.stats.stats` (usang, tidak dipakai) dihapus.

## 1. Setup: clone fork, install dependensi, build `ports_cpp`

In [ ]:
import os

REPO_URL = "https://github.com/aanpujianto070990/MEGA-GNN-TMLR.git"
REPO_DIR = "/kaggle/working/MEGA-GNN-TMLR"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Sesi lanjutan: update ke commit terbaru tanpa menghapus file untracked (.so, hasil run).
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main

assert os.path.isdir(f"{REPO_DIR}/MegaGNN"), "Clone gagal -- pastikan Settings -> Internet = On, lalu jalankan ulang sel ini."
# Sel-sel berikutnya memakai path absolut / cwd=REPO_DIR, jadi tidak bergantung pada %cd ini
# (efek %cd hilang kalau kernel di-restart).
%cd {REPO_DIR}
!git -C {REPO_DIR} log -1 --oneline

In [ ]:
import sys, torch

print("python", sys.version.split()[0])
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| gpu_available", torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("gpu", torch.cuda.get_device_name(0), "| compute capability", cap)
    if cap < (7, 0):
        print("PERINGATAN: GPU ini kemungkinan tidak didukung build PyTorch terbaru -- ganti Accelerator ke GPU T4.")

# Wheel PyG (pyg_lib, torch_scatter, torch_sparse) harus cocok dengan versi torch + CUDA.
TORCH_MM = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"
CUDA_TAG = ("cu" + torch.version.cuda.replace(".", "")) if torch.version.cuda else "cpu"
PYG_WHL = f"https://data.pyg.org/whl/torch-{TORCH_MM}+{CUDA_TAG}.html"
print("PyG wheels:", PYG_WHL)

!pip install -q torch_geometric yacs ogb gdown tensorboardX torchmetrics pybind11
# --only-binary: gagal cepat (bukan kompilasi 30+ menit) kalau wheel untuk versi torch ini belum ada.
!pip install -q --only-binary=:all: pyg_lib torch_scatter torch_sparse -f {PYG_WHL}
# genagg (vendored di repo); --no-deps supaya pip tidak mengutak-atik versi torch.
!pip install -q --no-deps -e {REPO_DIR}/genagg

In [ ]:
# Ekstensi C++ untuk port numbering (dipakai saat praproses semua dataset).
!cd {REPO_DIR}/MegaGNN/cpp && python setup.py build_ext --inplace > build.log 2>&1; tail -n 3 build.log; ls *.so

# Uji import seluruh paket (MegaGNN meng-import semua modul layer/dataset/train saat di-import).
!cd {REPO_DIR} && python -c "import MegaGNN, torch_scatter, torch_sparse; from MegaGNN.cpp import ports_cpp; print('Import MegaGNN OK')"

## 2. Data: tata letak folder yang diharapkan MEGA-GNN

MEGA-GNN membaca `<dataset.dir>/AML/<Small-HI|Small-LI>/raw/<HI|LI>-Small_Trans.csv`, lalu pada run pertama membuat `formatted_transactions.csv` (di folder `raw`) dan cache `processed/` (split, ports, pengelompokan sisi paralel). File mentah dari `/kaggle/input` cukup di-symlink.

In [ ]:
import glob

DATA_DIR = "/tmp/aml_data"                       # tidak ikut tersimpan di output Kaggle (besar)
OUT_DIR = "/kaggle/working/results_pilot"        # ringkasan & log run (ikut tersimpan)
os.makedirs(OUT_DIR, exist_ok=True)

DATASETS = {"Small-HI": "HI-Small_Trans.csv", "Small-LI": "LI-Small_Trans.csv"}

for name, fname in DATASETS.items():
    matches = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    if not matches:
        print(f"PERINGATAN: {fname} tidak ditemukan di /kaggle/input -- tambahkan dataset IBM AML sebagai Input.")
        continue
    raw_dir = f"{DATA_DIR}/AML/{name}/raw"
    os.makedirs(raw_dir, exist_ok=True)
    dst = f"{raw_dir}/{fname}"
    if not os.path.exists(dst):
        os.symlink(matches[0], dst)
    print(f"{name}: {matches[0]} -> {dst}")

## 3. Helper: menjalankan satu run dan meringkas hasilnya

`run_mega(model, dataset, seed, tag, overrides)` memanggil `python -m MegaGNN.main` dengan config asli paper, ditambah override:
- `wandb.use False` (tanpa API key), `gpu_mem True` (catat memori GPU per epoch), `out_dir`/`dataset.dir` ke folder Kaggle;
- `name_tag` supaya tiap tahap (smoke/bench/full) punya folder run sendiri: `<OUT_DIR>/<nama-config>-<tag>/<seed>/`.

Ringkasan diambil dari `train|val|test/stats.json` (satu baris JSON per epoch evaluasi) dan `logging.log`. Epoch terbaik dipilih seperti di `custom_train.py`: argmax **F1 validasi** di antara epoch evaluasi; metrik test dilaporkan pada epoch tersebut.

In [ ]:
import json, math, re, subprocess, threading, time
import numpy as np
import pandas as pd

REPO_DIR = "/kaggle/working/MEGA-GNN-TMLR"

CONFIGS = {
    ("Multi-GIN", "Small-HI"): "configs/GIN/AML-Small-HI-Multi-GIN.yaml",
    ("Multi-GIN", "Small-LI"): "configs/GIN/AML-Small-LI-Multi-GIN.yaml",
    ("MEGA-GIN", "Small-HI"): "configs/GIN/AML-Small-HI-MEGA-GIN.yaml",
    ("MEGA-GIN", "Small-LI"): "configs/GIN/AML-Small-LI-MEGA-GIN.yaml",
}

# F1 kelas ilegal (%) -- Bilgi & Atasu (TMLR 2026), Tabel 16: (rerata, simpangan baku)
PAPER_F1 = {
    ("Multi-GIN", "Small-HI"): (62.66, 1.73), ("Multi-GIN", "Small-LI"): (32.21, 0.99),
    ("MEGA-GIN", "Small-HI"): (70.83, 2.19), ("MEGA-GIN", "Small-LI"): (43.67, 0.55),
}


def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def summarize_run(run_dir, model, dataset, seed, tag, wall_s):
    tr = read_jsonl(f"{run_dir}/train/stats.json")
    va = read_jsonl(f"{run_dir}/val/stats.json")
    te = read_jsonl(f"{run_dir}/test/stats.json")
    log = open(f"{run_dir}/logging.log").read()

    best = int(np.argmax([r["f1"] for r in va]))  # sama dengan aturan custom_train.py
    v, t = va[best], te[best]
    params = re.search(r"Num parameters: (\d+)", log)
    avg_epoch = re.search(r"Avg time per epoch: ([0-9.]+)s", log)
    eval_s = float(np.mean([a["time_epoch"] + b["time_epoch"] for a, b in zip(va, te)]))
    n_epochs, n_evals = len(tr), len(va)
    avg_epoch_s = float(avg_epoch.group(1)) if avg_epoch else float("nan")

    summary = {
        "model": model, "dataset": dataset, "seed": seed, "tag": tag,
        "epochs_run": n_epochs, "evals_run": n_evals, "best_epoch": int(v["epoch"]),
        "val_f1": v["f1"] * 100, "test_f1": t["f1"] * 100,
        "test_precision": t["precision"] * 100, "test_recall": t["recall"] * 100,
        "test_pr_auc": t["pr-auc"] * 100, "test_roc_auc": t["auc"] * 100,
        "params": int(params.group(1)) if params else None,
        # Waktu dinding rata-rata per epoch (latih + evaluasi bila epoch itu dievaluasi).
        "avg_epoch_s": avg_epoch_s,
        # Satu evaluasi = val + test penuh.
        "eval_s": eval_s,
        # Waktu latih per epoch (256 iterasi) = total waktu epoch dikurangi waktu evaluasi.
        "train_epoch_s": (avg_epoch_s * n_epochs - n_evals * eval_s) / n_epochs,
        "gpu_mem_peak_mb": max((r.get("gpu_memory", 0) for r in tr + va + te), default=0),
        "wall_min": wall_s / 60,
        "run_dir": run_dir,
    }
    with open(f"{OUT_DIR}/summary_{model}_{dataset}_{tag}_s{seed}.json", "w") as f:
        json.dump(summary, f, indent=2)
    return summary


def poll_gpu_memory(proc, samples, interval=5):
    # Memori GPU 0 yang terpakai (MB) selama proses berjalan. Lebih andal daripada
    # gpu_memory di stats.json, yang bisa 0 kalau nvidia-smi tidak melihat PID di container.
    while proc.poll() is None:
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"], text=True)
            samples.append(int(out.strip().splitlines()[0]))
        except Exception:
            pass
        time.sleep(interval)


def run_mega(model, dataset, seed, tag, overrides=()):
    cfg = CONFIGS[(model, dataset)]
    cmd = [
        "python", "-u", "-m", "MegaGNN.main", "--cfg", cfg, "--gpu", "0",
        "seed", str(seed), "out_dir", OUT_DIR, "dataset.dir", DATA_DIR,
        "wandb.use", "False", "gpu_mem", "True", "name_tag", tag, *overrides,
    ]
    run_dir = f"{OUT_DIR}/{os.path.splitext(os.path.basename(cfg))[0]}-{tag}/{seed}"
    print("=" * 80 + "\n" + " ".join(cmd) + "\n" + "=" * 80)
    t0 = time.time()
    # cwd=REPO_DIR: `python -m MegaGNN.main` dan path config relatif harus dijalankan dari root repo.
    proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    gpu_samples = []
    poller = threading.Thread(target=poll_gpu_memory, args=(proc, gpu_samples), daemon=True)
    poller.start()
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    poller.join()
    if proc.returncode != 0:
        raise RuntimeError(f"Run gagal (exit {proc.returncode}) -- lihat log di atas / {run_dir}/logging.log")
    s = summarize_run(run_dir, model, dataset, seed, tag, time.time() - t0)
    s["gpu_mem_peak_mb"] = max([s["gpu_mem_peak_mb"], *gpu_samples])
    with open(f"{OUT_DIR}/summary_{model}_{dataset}_{tag}_s{seed}.json", "w") as f:
        json.dump(s, f, indent=2)
    print(f"\nSelesai {s['wall_min']:.1f} menit | test F1 {s['test_f1']:.2f}% (best val epoch {s['best_epoch']})"
          f" | params {s['params']} | GPU puncak {s['gpu_mem_peak_mb']} MB")
    return s

## Tahap A - smoke test

Run pertama pada HI-Small sekaligus memicu praproses (format CSV, split, ports, pengelompokan sisi paralel); bagian `find_parallel_edges` memakai loop Python sehingga bisa makan beberapa menit. Epoch dipersingkat jadi 4 iterasi, tetapi evaluasi tetap menyapu seluruh val/test.

In [ ]:
SMOKE = ["optim.max_epoch", "1", "train.iter_per_epoch", "4", "train.eval_period", "1"]

smoke = [run_mega(m, "Small-HI", seed=1, tag="smoke", overrides=SMOKE) for m in ["Multi-GIN", "MEGA-GIN"]]
pd.DataFrame(smoke)[["model", "dataset", "params", "eval_s", "gpu_mem_peak_mb", "wall_min"]]

### Cek hasil praproses terhadap statistik dataset

Membandingkan `formatted_transactions.csv` hasil formatter fork dengan angka di proposal (Tabel 2.1): HI-Small 5.078.345 transaksi / 5.177 ilegal; LI-Small 6.924.049 / 3.565. Jumlah rekening di proposal (515.088 / 705.907) berasal dari FraudGT; di sini dihitung sebagai jumlah id unik pengirim/penerima. Split hari yang dipakai tercetak di log run sebagai `Calculate split`.

In [ ]:
EXPECTED = {"Small-HI": (5_078_345, 5_177), "Small-LI": (6_924_049, 3_565)}

for name in DATASETS:
    path = f"{DATA_DIR}/AML/{name}/raw/formatted_transactions.csv"
    if not os.path.exists(path):
        print(f"{name}: belum diproses (akan dibuat di run pertama dataset ini)")
        continue
    df = pd.read_csv(path)
    n_tx, n_ill = len(df), int(df["Is Laundering"].sum())
    n_acc = int(pd.concat([df["from_id"], df["to_id"]]).nunique())
    exp_tx, exp_ill = EXPECTED[name]
    ok = "OK" if (n_tx, n_ill) == (exp_tx, exp_ill) else "BEDA -- periksa!"
    print(f"{name}: {n_tx:,} transaksi, {n_ill:,} ilegal ({n_ill / n_tx * 100:.3f}%), {n_acc:,} rekening, "
          f"hari {df['Timestamp'].max() / 86400:.1f} -> {ok}")
    print(df.head(3).to_string(index=False))

## Tahap B - benchmark waktu & memori

Tiap kombinasi dijalankan 4 epoch dengan `eval_period 4` (evaluasi di epoch 0 dan 3) memakai config asli. Hasilnya dipakai untuk memproyeksikan dua protokol:
- **Paper**: 400 epoch x 256 iterasi (batch 2048), evaluasi tiap 4 epoch (101 evaluasi).
- **Proposal**: maks. 20 epoch, satu epoch = satu putaran penuh seed edge latih, evaluasi tiap epoch (skenario terburuk tanpa early stopping).

Total proposal = 140 run: Multi-GIN 5 konfigurasi x 10 seed x 2 dataset (100) + MEGA-GIN 2 konfigurasi x 10 seed x 2 dataset (40). Overhead modul temporal belum dihitung.

In [ ]:
BENCH = ["optim.max_epoch", "4", "train.eval_period", "4"]

bench = []
for dataset in ["Small-HI", "Small-LI"]:
    for model in ["Multi-GIN", "MEGA-GIN"]:
        bench.append(run_mega(model, dataset, seed=1, tag="bench", overrides=BENCH))

pd.DataFrame(bench)[["model", "dataset", "params", "train_epoch_s", "eval_s", "gpu_mem_peak_mb", "wall_min"]]

In [ ]:
BATCH_SIZE, ITERS_PER_EPOCH_PAPER = 2048, 256
RUNS_PROPOSAL = {"Multi-GIN": 5 * 10, "MEGA-GIN": 2 * 10}   # per dataset
KAGGLE_GPU_H_PER_WEEK = 30


def n_train_edges(dataset):
    d = torch.load(f"{DATA_DIR}/AML/{dataset}/processed/data.pt", weights_only=False)
    return int(d["train"]["node", "to", "node"].split_mask.sum())


rows = []
for s in bench:
    n_tr = n_train_edges(s["dataset"])
    iters_full = math.ceil(n_tr / BATCH_SIZE)
    t_iter = s["train_epoch_s"] / ITERS_PER_EPOCH_PAPER
    paper_h = (400 * s["train_epoch_s"] + 101 * s["eval_s"]) / 3600
    proposal_h = 20 * (iters_full * t_iter + s["eval_s"]) / 3600
    rows.append({
        "model": s["model"], "dataset": s["dataset"], "train_edges": n_tr, "iters_full_epoch": iters_full,
        "jam_per_run_paper": round(paper_h, 2), "jam_per_run_proposal_maks": round(proposal_h, 2),
        "run_proposal": RUNS_PROPOSAL[s["model"]],
        "jam_total_proposal_maks": round(proposal_h * RUNS_PROPOSAL[s["model"]], 1),
    })

proj = pd.DataFrame(rows)
total_h = proj["jam_total_proposal_maks"].sum()
print(f"Proyeksi protokol proposal (140 run, tanpa early stopping): {total_h:.0f} jam GPU "
      f"~ {total_h / KAGGLE_GPU_H_PER_WEEK:.1f} minggu kuota Kaggle ({KAGGLE_GPU_H_PER_WEEK} jam/minggu)")
too_long = proj[proj["jam_per_run_paper"] > 11]
if len(too_long):
    print("PERINGATAN: reproduksi protokol paper > 11 jam (batas sesi Kaggle 12 jam) untuk:",
          ", ".join(too_long["model"] + "/" + too_long["dataset"]))
proj

## Tahap C - reproduksi penuh (config asli paper)

Jalankan **satu kombinasi per sesi** (sesi Kaggle maksimal 12 jam). Untuk run panjang, pakai *Save Version -> Save & Run All (Commit)* agar berjalan di latar belakang dan hasilnya tersimpan; ubah `MODEL`/`DATASET` lalu commit lagi untuk kombinasi berikutnya. Lewati sel Tahap A/B pada sesi itu kalau sudah pernah dijalankan (atau biarkan; smoke test hanya beberapa menit).

Kalau proyeksi Tahap B menunjukkan > 11 jam, jangan dijalankan apa adanya -- diskusikan dulu (mis. GPU lain, atau langsung ke protokol proposal).

In [ ]:
MODEL = "MEGA-GIN"      # "Multi-GIN" atau "MEGA-GIN"
DATASET = "Small-LI"    # "Small-HI" atau "Small-LI"
SEED = 1

full = run_mega(MODEL, DATASET, seed=SEED, tag="full")

## Ringkasan: hasil pilot vs paper

In [ ]:
summaries = [json.load(open(p)) for p in sorted(glob.glob(f"{OUT_DIR}/summary_*_full_s*.json"))]
if not summaries:
    print("Belum ada run Tahap C.")
else:
    res = pd.DataFrame(summaries)
    res["paper_f1"] = [PAPER_F1[(m, d)][0] for m, d in zip(res["model"], res["dataset"])]
    res["paper_std"] = [PAPER_F1[(m, d)][1] for m, d in zip(res["model"], res["dataset"])]
    res["selisih_f1"] = res["test_f1"] - res["paper_f1"]
    display(res[["model", "dataset", "seed", "best_epoch", "test_f1", "paper_f1", "paper_std", "selisih_f1",
                 "test_precision", "test_recall", "test_pr_auc", "params", "gpu_mem_peak_mb", "wall_min"]].round(2))